# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR² Croissant](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`. The metadata object provides summary information, and the records iterator returns records per record set.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
print(f"Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Keywords: {', '.join(dataset.metadata.keywords)}")

## 2. Data Overview

Let's inspect available record sets and print their `@id`, along with available field IDs and descriptions. You can use these identifiers in downstream code blocks.

> **Note:** All entities must be referenced by their `@id`.


In [ ]:
# Print all available record sets and their field ids
print('Record Sets:')
for rs in dataset.record_sets:
    print(f"  RecordSet @id: {rs['@id']}, name: {rs.get('name', '-')}")
    # Print Fields
    if 'field' in rs:
        print('    Fields:')
        for field in rs['field']:
            # To get actual field info, map field dict or id to metadata field
            if isinstance(field, dict):
                fid = field.get('@id', field)
            else:
                fid = field
            fmeta = next((f for f in dataset.fields if f['@id']==fid), None)
            if fmeta:
                fname = fmeta.get('name', '-')
                fdesc = fmeta.get('description', '-')
                print(f"      - Field @id: {fmeta['@id']}, name: {fname}, desc: {fdesc}")
            else:
                print(f"      - Field @id: {fid}")


## 3. Data Extraction

Load data from one or more record sets into pandas DataFrames for analysis. **Reference all record sets and fields by their `@id` only.**

Below, we extract all available record sets.

In [ ]:
# Build a list of all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# Map @id -> DataFrame with rows from each record set
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading records from RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Fields (@id) in this RecordSet: {list(df.columns)}\n")
        print(df.head(2))
    else:
        print(f"No records found for {record_set_id}.")

# For convenience, pick one record_set_id for demo if available:
if record_set_ids:
    example_record_set_id = record_set_ids[0]
else:
    example_record_set_id = None

## 4. Exploratory Data Analysis (EDA)

We'll proceed with EDA using a numeric field from the chosen record set (by `@id`). To do so, first, we identify a numeric field.

> **Tip:** Replace the `numeric_field_id` and `group_field_id` below with the relevant field `@id`s for your analysis. All field references are by `@id`.

In [ ]:
# Example: Choose a numeric field for EDA
# (Replace these with your actual field @ids as found earlier)
numeric_field_id = None  # e.g., 'cr:coverage' or any numeric field in this dataset
group_field_id = None    # e.g., 'cr:sample_name' or a categorical field

# Try to auto-select a numeric field by scanning the DataFrame
df = dataframes.get(example_record_set_id)
if df is not None:
    # Find numeric columns
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"Auto-selected numeric field @id: {numeric_field_id}")
    else:
        print("No numeric field found in example record set.")
    # Try to auto-select a group field
    object_cols = df.select_dtypes(include=['object']).columns.tolist()
    if object_cols:
        group_field_id = object_cols[0]
        print(f"Auto-selected group field @id: {group_field_id}")

    # Filtering step
    threshold = df[numeric_field_id].mean() if numeric_field_id else 0
    print(f"Applying threshold: {numeric_field_id} > {threshold}")
    filtered_df = df[df[numeric_field_id] > threshold].copy() if numeric_field_id else df.copy()
    print(f"Filtered DataFrame (first 5 rows):\n", filtered_df.head())

    # Normalizing
    if numeric_field_id:
        filtered_df[f'{numeric_field_id}_normalized'] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} (first 5 rows):")
        print(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

    # Grouping step
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data (mean {numeric_field_id} by {group_field_id}):")
        print(grouped_df.head())
else:
    print("No records available to perform EDA.")

## 5. Visualization

Visualize the distribution of the chosen numeric field and examine group-wise statistics if relevant.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=45)
        plt.title(f'{numeric_field_id} across {group_field_id}')
        plt.show()
else:
    print("Visualization not available due to missing data or numeric field.")

## 6. Conclusion

* This notebook demonstrates how to load and explore a [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) with `mlcroissant`, referencing all key entities by their `@id` as best practice.
* Steps shown include: discovering record set and field ids, extracting data, basic EDA (filtering, normalization, grouping), and producing visualizations.
* For more advanced usage, consult [`mlcroissant` documentation](https://github.com/mlcommons/croissant) and adapt field and record set references by their `@id`.